In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import os
import scipy.stats as stats
from scipy.special import logsumexp
import pandas as pd
import matplotlib.cm as cm
import math
import itertools
import random
from matplotlib.colors import TwoSlopeNorm, LinearSegmentedColormap
from random import random

from scipy.stats import linregress
from sklearn.linear_model import RANSACRegressor, LinearRegression
from scipy.stats import chi2_contingency, fisher_exact

import sys
sys.path.append('../src')

from em_clustering import (
    recursive_em_split, 
    reassign_cells
)

from visualization import (
    plot_cost_function_with_minima, 
    plot_ploidy_optimization,
    plot_ploidy_comparison,  
    plot_overlap_matrix_figure,
    plot_uniformity_histograms_nice
)
from copy_number import (
    make_unif_dict,
    compute_read_counts_relative_to_normal, 
    compute_average_segment_counts, 
    compute_optimal_ploidy_and_profile,
    create_copy_number_matrix_fast
)

import time

In [ ]:
# 1. Load Data
tumor ="t57"
infilename = f"../data/{tumor}/{tumor}.bin_count_table_all_cells.txt"
bincounts = pd.read_csv(infilename, sep="\t")
bincounts_no_WGS = pd.read_csv(f"../data/{tumor}/{tumor}.bincounts_without_WGS_segmentation.txt", sep="\t")

In [ ]:
bincounts.head()

In [ ]:
bincounts_no_WGS.head()

In [ ]:
# Use ALL cells instead of filtered cells
all_cell_names = [col for col in bincounts.columns if col not in ['chrom', 'abstart', 'abend', 'lengths','loc.start','loc.end']]
cells = pd.read_csv(f"../data/{tumor}/{tumor}.not_broken_final_list.txt", header=None)[0].values


In [ ]:
# Prepare data
data = bincounts[["chrom", "abstart", "abend"] + all_cell_names].copy()
data["lengths"] = data["abend"] - data["abstart"]
all_data = data[cells].copy()
data_no_WGS = bincounts_no_WGS[["chrom", "abstart", "abend"] + all_cell_names].copy()
data_no_WGS["lengths"] = data_no_WGS["abend"] - data_no_WGS["abstart"]
all_data_no_WGS = data_no_WGS[cells].copy()
with_broken = data[all_cell_names].copy()


In [ ]:
all_data.head()

In [ ]:
# 2. Run Recursive EM Splitting (two independent runs)
print("Running recursive EM split A...")
final_populations_A = recursive_em_split(
    all_data, cells, 
    threshold=3/4, max_depth=10, name="A"
)

In [ ]:
final_populations_B = recursive_em_split(
    all_data_no_WGS, cells, 
    threshold=3/4, max_depth=10, name="B"
)

In [ ]:
# 3. Reassign cells based on multinomial parameters
A_dict = reassign_cells(all_data, final_populations_A)
B_dict = reassign_cells(all_data_no_WGS, final_populations_B)


In [ ]:
final_populations_C = recursive_em_split(
    with_broken, all_cell_names, 
    threshold=3/4, max_depth=10, name="C"
)

In [ ]:
C_dict = reassign_cells(with_broken, final_populations_C)

In [ ]:
overlap_df = plot_overlap_matrix_figure(A_dict, B_dict)

In [ ]:
# # Save population assignments A_dict to a text file
# output_file = f"../data/{tumor}/{tumor}_population_assignments_A_WGS_bins.txt"
# with open(output_file, 'w') as f:
#     #header
#     f.write("cell\tpopulation\n")
#     for pop_name, details in A_dict.items():
#         cell_list = details['cells']
#         for cell in cell_list:
#             f.write(f"{cell}\t{pop_name}\n")

# #Save population assignments B_dict to a text file
# output_file = f"../data/{tumor}/{tumor}_population_assignments_B_equi_bins.txt"
# with open(output_file, 'w') as f:
#     #header
#     f.write("cell\tpopulation\n")
#     for pop_name, details in B_dict.items():
#         cell_list = details['cells']
#         for cell in cell_list:
#             f.write(f"{cell}\t{pop_name}\n")

#Save population assignments C_dict to a text file
output_file = f"../data/{tumor}/{tumor}_population_assignments_C_WGS_with_broken_cells.txt"
with open(output_file, 'w') as f:
    #header
    f.write("cell\tpopulation\n")
    for pop_name, details in C_dict.items():
         cell_list = details['cells']
         for cell in cell_list:
             f.write(f"{cell}\t{pop_name}\n")